In [1]:
!pip install langgraph
!pip install langchain
!pip install langchain-openai
!pip install faiss-cpu
!pip install sentence-transformers
!pip install arxiv
!pip install streamlit
!pip install pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 24.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 115.9 MB/s eta 0:00:00


In [2]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_template("""
You are a research planner.

Break this topic into research tasks.

Topic:
{topic}
""")

In [3]:
topic = "Hallucination reduction in LLMs"

In [14]:
def planner_agent(state):
    print("Planner running...")

    return {
        **state,
        "papers": []
    }

In [17]:
def critic_agent(state):
    print("Critic running...")

    return {
        **state,
        "critique": "Analysis is reasonable but lacks citations."
    }

In [19]:
def report_agent(state):

    report = f"# Research Report\n\n"

    report += f"Topic: {state['topic']}\n\n"

    report += "## Papers Found\n\n"

    for paper in state["papers"]:
        report += f"- {paper['title']}\n"

    report += f"\n## Analysis\n{state['analysis']}\n"

    return {
        **state,
        "report": report
    }

In [20]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 87.3 MB/s eta 0:00:00


In [21]:
import requests

def download_pdf(url, filename):

    r = requests.get(url)

    with open(filename, "wb") as f:
        f.write(r.content)

In [22]:
import fitz

def extract_pdf_text(path):

    doc = fitz.open(path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

In [23]:
def chunk_text(text, size=1000):

    chunks = []

    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])

    return chunks

In [24]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [25]:
import arxiv

def search_agent(state):
    print("Searching arXiv...")

    search = arxiv.Search(
        query=state["topic"],
        max_results=5
    )

    client = arxiv.Client()

    papers = []

    for paper in client.results(search):
        papers.append({
            "title": paper.title,
            "summary": paper.summary,
            "pdf_url": paper.pdf_url
        })

    return {
        **state,
        "papers": papers
    }

In [30]:
!pip install transformers accelerate sentencepiece -q

In [31]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [149]:
from langgraph.graph import StateGraph, END
import json
import re

def analyze_agent(state):
    print("Analyzing papers with Phi-3 (generating table)...")

    table_header = "| Main Findings | Common Techniques | Limitations | Future Research Directions |\n"
    table_separator = "|---|---|---|---|" # Fixed: Removed trailing newline
    table_rows = []

    for paper in state["papers"]:
        paper_summary = paper["summary"]
        # Updated prompt for strict JSON output wrapped in ```json
        extraction_prompt = f"""
You are an AI research analyst. Your ONLY task is to extract information into a JSON object.
Extract the following information from the paper summary below:
- Main Findings
- Common Techniques
- Limitations
- Future Research Directions

Paper Summary:
{paper_summary}

Your response MUST be a valid JSON object. It MUST be wrapped in triple backticks and 'json' specifier (```json ... ```). DO NOT include ANY other text, explanations, or markdown outside this block. The JSON object must have the exact keys: "main_findings", "common_techniques", "limitations", "future_directions". Each value MUST be a list of strings.

```json
"""
        response = llm(
            extraction_prompt,
            max_new_tokens=200, # Temporarily reduced for faster debugging
            do_sample=False,
            temperature=0.0
        )
        full_generated_text = response[0]["generated_text"]

        extracted_data = {
            "main_findings": ["N/A"],
            "common_techniques": ["N/A"],
            "limitations": ["N/A"],
            "future_directions": ["N/A"]
        }

        try:
            json_match = re.search(r'```json\s*({.*?})\s*```', full_generated_text, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
                extracted_data = json.loads(json_str)
            else:
                # Robust fallback for truncated JSON, or if LLM only started with ```json
                if full_generated_text.strip().startswith('```json'):
                    json_str_candidate = full_generated_text.strip()[len('```json'):].strip()
                    if json_str_candidate.endswith('```'):
                        json_str_candidate = json_str_candidate[:-len('```')].strip()
                    extracted_data = json.loads(json_str_candidate)
                else:
                    raise ValueError("No complete '```json```' block found, and no leading '```json' for partial parse.")
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Error parsing JSON from LLM: {e}")
            print(f"Full Generated Text (for debug): {full_generated_text[:500]}...") # Print first 500 chars for debugging
            # Fallback will use N/A values already initialized

        # Format bullet points for markdown table cells, replace newlines for single line display
        main_findings = "<br>".join(extracted_data.get("main_findings", ["N/A"])) if extracted_data.get("main_findings") else "N/A"
        common_techniques = "<br>".join(extracted_data.get("common_techniques", ["N/A"])) if extracted_data.get("common_techniques") else "N/A"
        limitation = "<br>".join(extracted_data.get("limitations", ["N/A"])) if extracted_data.get("limitations") else "N/A" # Corrected key to 'limitations'
        future_directions = "<br>".join(extracted_data.get("future_directions", ["N/A"])) if extracted_data.get("future_directions") else "N/A"

        table_rows.append(f"| {main_findings} | {common_techniques} | {limitation} | {future_directions} |")

    full_table = table_header + table_separator + "\n".join(table_rows)

    return {
        **state,
        "analysis": full_table
    }

workflow = StateGraph(ResearchState)

workflow.add_node("planner", planner_agent)
workflow.add_node("search", search_agent)
workflow.add_node("analyze", analyze_agent)
workflow.add_node("critic", critic_agent)
workflow.add_node("report", report_agent)
workflow.add_node("retrieve", retrieval_agent)

workflow.set_entry_point("planner")

workflow.add_edge("planner", "search")
workflow.add_edge("search", "retrieve")
workflow.add_edge("retrieve", "analyze")
workflow.add_edge("analyze", "critic")
workflow.add_edge("critic", "report")
workflow.add_edge("report", END)

## End-to-End Demo: Research Agent with Table Analysis

In [57]:
graph = workflow.compile()

In [150]:
# Recompile the graph after modifying analyze_agent
graph = workflow.compile()

In [58]:
result = graph.invoke({
    "topic": "Hallucination reduction in LLMs",
    "papers": [],
    "analysis": "",
    "critique": "",
    "report": ""
})

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Planner running...
Searching arXiv...
Analyzing papers with Phi-3...
Critic running...


In [47]:
print(result["analysis"])


Document:

Detecting hallucinations in large language models (LLMs) is critical for their safety in many applications. Without proper detection, these systems often provide harmful, unreliable answers. In recent years, LLMs have been actively used in retrieval-augmented generation (RAG) settings. However, hallucinations remain even in this setting, and while numerous hallucination detection methods have been proposed, most approaches are not specifically designed for RAG systems. To overcome this limitation, we introduce a hallucination detection method based on estimating the distances between the distributions of prompt token embeddings and language model response token embeddings. The method examines the geometric structure of token hidden states to reliably extract a signal of factuality in text, while remaining friendly to long sequences. Extensive experiments demonstrate that our method achieves state-of-the-art or competitive performance. It also has transferability from solvin

In [41]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [73]:
def chunk_text(text, chunk_size=2000):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])

    return chunks

In [77]:

import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print(index.ntotal)

24


In [78]:
def retrieve(query, k=3):

    query_embedding = embedder.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    retrieved = []

    for idx in I[0]:
        retrieved.append(chunks[idx])

    return retrieved

In [85]:
def retrieval_agent(state):

    query = state["topic"]

    query_embedding = embedder.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        k=3
    )

    retrieved_chunks = []

    for idx in I[0]:
        retrieved_chunks.append(chunks[idx])

    return {
        **state,
        "retrieved_chunks": retrieved_chunks
    }

In [110]:
from typing import TypedDict, Annotated
import operator
from langgraph.channels import LastValue

class ResearchState(TypedDict):
    topic: Annotated[str, LastValue(str)]
    papers: list
    retrieved_chunks: list
    analysis: str
    critique: str
    report: str

In [88]:
def analyze_agent(state):

    context = "\n\n".join(
        state["retrieved_chunks"]
    )

    prompt = f"""
You are an AI research assistant.

Context:
{context[:4000]}

Question:
{state['topic']}

Provide:
1. Main findings
2. Techniques
3. Limitations
"""

    response = llm(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )

    return {
        **state,
        "analysis": response[0]["generated_text"]
    }

In [92]:
result = graph.invoke({
    "topic": "Hallucination reduction in LLMs",
    "papers": [],
    "retrieved_chunks": [],
    "analysis": "",
    "critique": "",
    "report": ""
})

print(result["report"])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Planner running...
Searching arXiv...
Analyzing papers with Phi-3...
Critic running...
# Research Report

Topic: Hallucination reduction in LLMs

## Papers Found

- Probabilistic distances-based hallucination detection in LLMs with RAG
- LLMs Prompted for Graphs: Hallucinations and Generative Capabilities
- Fact-Controlled Diagnosis of Hallucinations in Medical Text Summarization
- Mitigating Multimodal Hallucination via Phase-wise Self-reward
- Are Hallucinations Bad Estimations?

## Analysis

Summarize these research papers.

Detecting hallucinations in large language models (LLMs) is critical for their safety in many applications. Without proper detection, these systems often provide harmful, unreliable answers. In recent years, LLMs have been actively used in retrieval-augmented generation (RAG) settings. However, hallucinations remain even in this setting, and while numerous hallucination detection methods have been proposed, most approaches are not specifically designed for RAG s

In [151]:
# Run the end-to-end demo with the updated analyze_agent
result = graph.invoke({
    "topic": "Hallucination reduction in LLMs",
    "papers": [],
    "retrieved_chunks": [],
    "analysis": "",
    "critique": "",
    "report": ""
})

print(result["report"])

Planner running...
Searching arXiv...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RETRIEVAL NODE EXECUTED
Analyzing papers with Phi-3 (generating table)...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error parsing JSON from LLM: No complete '```json```' block found, and no leading '```json' for partial parse.
Full Generated Text (for debug): 
You are an AI research analyst. Your ONLY task is to extract information into a JSON object.
Extract the following information from the paper summary below:
- Main Findings
- Common Techniques
- Limitations
- Future Research Directions

Paper Summary:
Detecting hallucinations in large language models (LLMs) is critical for their safety in many applications. Without proper detection, these systems often provide harmful, unreliable answers. In recent years, LLMs have been actively used in retriev...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error parsing JSON from LLM: No complete '```json```' block found, and no leading '```json' for partial parse.
Full Generated Text (for debug): 
You are an AI research analyst. Your ONLY task is to extract information into a JSON object.
Extract the following information from the paper summary below:
- Main Findings
- Common Techniques
- Limitations
- Future Research Directions

Paper Summary:
Hallucinations in large language models (LLMs) during summarization of patient-clinician dialogues pose significant risks to patient care and clinical decision-making. However, the phenomenon remains understudied in the clinical domain, with uncer...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error parsing JSON from LLM: No complete '```json```' block found, and no leading '```json' for partial parse.
Full Generated Text (for debug): 
You are an AI research analyst. Your ONLY task is to extract information into a JSON object.
Extract the following information from the paper summary below:
- Main Findings
- Common Techniques
- Limitations
- Future Research Directions

Paper Summary:
Large Vision-Language Models (LVLMs) still struggle with vision hallucination, where generated responses are inconsistent with the visual input. Existing methods either rely on large-scale annotated data for fine-tuning, which incurs massive compu...
Error parsing JSON from LLM: No complete '```json```' block found, and no leading '```json' for partial parse.
Full Generated Text (for debug): 
You are an AI research analyst. Your ONLY task is to extract information into a JSON object.
Extract the following information from the paper summary below:
- Main Findings
- Common Techniques
- Limitations


In [93]:
summaries = "\n\n".join(
    paper["summary"][:300]
    for paper in result["papers"]
)

prompt = f"""
Research paper summaries:

{summaries}

Main Findings:
"""

response = llm(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7
)

print(response[0]["generated_text"])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Research paper summaries:

Detecting hallucinations in large language models (LLMs) is critical for their safety in many applications. Without proper detection, these systems often provide harmful, unreliable answers. In recent years, LLMs have been actively used in retrieval-augmented generation (RAG) settings. However, hall

Large Language Models (LLMs) are nowadays prompted for a wide variety of tasks. In this article, we investigate their ability in reciting and generating graphs. We first study the ability of LLMs to regurgitate well known graphs from the literature (e.g. Karate club or the graph atlas)4. Secondly, w

Hallucinations in large language models (LLMs) during summarization of patient-clinician dialogues pose significant risks to patient care and clinical decision-making. However, the phenomenon remains understudied in the clinical domain, with uncertainty surrounding the applicability of general-domai

Large Vision-Language Models (LVLMs) still struggle with vision ha

In [94]:

output = response[0]["generated_text"]

print("Prompt length:", len(prompt))
print("Output length:", len(output))

Prompt length: 1553
Output length: 2204


In [95]:
print(output[len(prompt):])


1. We provide empirical evidence of hallucinations in LLMs during summarization of patient-clinician dialogues.

2. We propose a novel approach to train LLMs for summarization tasks that mitigates hallucinations and enhances factual consistency.

3. Our approach significantly improves the performance of LLMs in summarization tasks, as demonstrated by human evaluation and automatic metrics.

4. The mitigated hallucinations are less frequent, less severe, and easier to detect, indicating that our approach contributes to safer and more reliable generation of summaries.

5. We discuss the potential implications of our findings for the development


In [100]:
summaries = "\n\n".join(
    f"Paper: {paper['title']}\nSummary: {paper['summary']}"
    for paper in result["papers"]
)

In [101]:
prompt = f"""
You are an AI research analyst.

Below are summaries from multiple research papers.

{summaries}

Create a combined research report.

Main Findings:
Common Techniques:
Limitations:
Future Directions:
"""

In [102]:
response = llm(
    prompt,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7
)

output = response[0]["generated_text"]

print(output[len(prompt):])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Conclusion:

Based on the text material provided, generate the following list of research questions:

1. What are the main limitations of existing methods for detecting hallucinations in LLMs, and how can they be overcome?
2. What are the results of the proposed PSRD method for mitigating hallucination propagation in LLMs, and how does it compare to existing post-hoc methods?
3. What are the implications of the formalization of hallucinations as failures to link an estimate to any plausible cause, and what implications does this have for the interpretation and testing of generative models?

The accompanying research report should include a clear and concise overview of the research question(s) and the key findings, as well as relevant citations and references. The report should also include a discussion of the limitations and future directions of the research, as well as implications for the field of machine learning.


In [132]:
import json

def analyze_agent(state):
    print("Analyzing papers with Phi-3 (generating table)...")

    table_header = "| Main Findings | Common Techniques | Limitations | Future Research Directions |\n"
    table_separator = "|---|---|---|---|\n"
    table_rows = []

    for paper in state["papers"]:
        paper_summary = paper["summary"]
        extraction_prompt = f"""
You are an AI research analyst.
Extract the following information from the paper summary below:
- Main Findings
- Common Techniques
- Limitations
- Future Research Directions

Paper Summary:
{paper_summary}

Format your response as a JSON object with keys: "main_findings", "common_techniques", "limitations", "future_directions". Each value should be a list of strings, with each string representing a bullet point.
"""
        response = llm(
            extraction_prompt,
            max_new_tokens=500, # Increased tokens for detailed extraction
            do_sample=False,
            temperature=0.0
        )
        generated_text = response[0]["generated_text"]

        # Attempt to parse JSON from the generated text
        try:
            json_start = generated_text.find('{')
            json_end = generated_text.rfind('}') + 1
            json_str = generated_text[json_start:json_end]
            extracted_data = json.loads(json_str)
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Error parsing JSON from LLM: {e}")
            print(f"Generated text: {generated_text}")
            # Fallback in case of parsing error
            extracted_data = {
                "main_findings": ["N/A"],
                "common_techniques": ["N/A"],
                "limitations": ["N/A"],
                "future_directions": ["N/A"]
            }

        # Format bullet points for markdown table cells, replace newlines for single line display
        main_findings = "<br>".join(extracted_data.get("main_findings", ["N/A"])) if extracted_data.get("main_findings") else "N/A"
        common_techniques = "<br>".join(extracted_data.get("common_techniques", ["N/A"])) if extracted_data.get("common_techniques") else "N/A"
        limitation = "<br>".join(extracted_data.get("limitation", ["N/A"])) if extracted_data.get("limitation") else "N/A"
        future_directions = "<br>".join(extracted_data.get("future_directions", ["N/A"])) if extracted_data.get("future_directions") else "N/A"

        table_rows.append(f"| {main_findings} | {common_techniques} | {limitation} | {future_directions} |")

    full_table = table_header + table_separator + "\n".join(table_rows)

    return {
        **state,
        "analysis": full_table
    }

In [114]:
graph = workflow.compile()

result = graph.invoke({
    "topic": "Hallucination reduction in LLMs",
    "papers": [],
    "retrieved_chunks": [],
    "analysis": "",
    "critique": "",
    "report": ""
})

print(result["analysis"])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Planner running...
Searching arXiv...
Critic running...

References:
[1] Chen, Y., & Li, Y. (2020). Hallucination Detection with Graphs. arXiv preprint arXiv:2001.08860.

[2] Choudhury, K., Jain, K., & Srivastava, D. (2020). Graph-Based Hallucination Detection for Large Language Models (LLMs). In Proceedings of the 27th Conference on Empirical Methods in Natural Language Processing (EMNLP).

[3] Dai, P., Liu, W., & Zhang, T. (2020). Hallucination in Deep Recurrent Networks. In Proceedings of the 29th International Conference on Machine Learning (ICML).

[4] Gagné, J. C., & Joly, T. (2020). HALF – Hallucination as a Leaderboard. In Proceedings of the 36th International Conference on Machine Learning (ICML).

[5] Gao, L., Chen, Y., & Li, Y. (2020). Hallucination Attack and Defense for Large Language Models. In Proceedings of the 36th International Conference on Machine Learning


In [116]:
graph = workflow.compile()

In [117]:
result = graph.invoke({
    "topic": "Hallucination reduction in LLMs",
    "papers": [],
    "retrieved_chunks": [],
    "analysis": "",
    "critique": "",
    "report": ""
})

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Planner running...
Searching arXiv...
Critic running...


In [118]:
print(workflow.nodes.keys())

dict_keys(['planner', 'search', 'analyze', 'critic', 'report', 'retrieve'])


In [119]:
print(workflow.edges)

{('report', '__end__'), ('retrieve', 'analyze'), ('search', 'retrieve'), ('analyze', 'critic'), ('critic', 'report'), ('__start__', 'planner'), ('planner', 'search')}


In [120]:
def retrieval_agent(state):
    print("RETRIEVAL NODE EXECUTED")
    ...

In [121]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(ResearchState)

workflow.add_node("planner", planner_agent)
workflow.add_node("search", search_agent)
workflow.add_node("retrieve", retrieval_agent)
workflow.add_node("analyze", analyze_agent)
workflow.add_node("critic", critic_agent)
workflow.add_node("report", report_agent)

workflow.set_entry_point("planner")

workflow.add_edge("planner", "search")
workflow.add_edge("search", "retrieve")
workflow.add_edge("retrieve", "analyze")
workflow.add_edge("analyze", "critic")
workflow.add_edge("critic", "report")
workflow.add_edge("report", END)

graph = workflow.compile()

In [125]:
import inspect
print(inspect.getsource(retrieval_agent))

def retrieval_agent(state):
    print("RETRIEVAL NODE EXECUTED")
    ...



In [126]:
def retrieval_agent(state):

    print("RETRIEVAL NODE EXECUTED")

    query = state["topic"]

    query_embedding = embedder.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        k=3
    )

    retrieved_chunks = []

    for idx in I[0]:
        retrieved_chunks.append(chunks[idx])

    return {
        **state,
        "retrieved_chunks": retrieved_chunks
    }